### RT Whisper 하이퍼파라미터 리스트 평가

In [ ]:
import os
import sys
os.chdir("/workspaces/dev")
paths = [
    "/workspaces/dev/test/performance_test",
]
for path in paths:
    sys.path.append(os.path.abspath(path))
print(f"Current Python version: {sys.version}")


In [ ]:
from pathlib import Path

In [ ]:
from sj_ai_utils.datasets.l_hotse import LibriSpeech

In [ ]:
from common_util import evaluate

In [ ]:
SEED = 42
SAMPLE_SIZE = -1
TEST_ALL = True
USE_TOKEN_SAVER_LOADER = True
USE_PROMPT = False
CHUNK_SIZE = 48000

SOURCE = "/workspaces/dev/.datasets/libri_speech"
STORAGE = "/workspaces/dev/.storage/libri/"
HYPERPARAMETER = "/workspaces/dev/test/optimize/all/hyperparameters/20250905/3s/step1_3s-96k-cpm-hyper"
OUTPUT_PATH = "/workspaces/dev/test/performance_test/libri/output/dev-clean/20250905/step1_3s-96k-cpm-hyper"

In [ ]:
source = Path(SOURCE)
storage = Path(STORAGE)
hyperparameter_path = Path(HYPERPARAMETER)
output_path = Path(OUTPUT_PATH)

if not source.exists():
    raise FileNotFoundError(f"Source path does not exist: {source}")
if not hyperparameter_path.exists():
    raise FileNotFoundError(f"Hyperparameter path does not exist: {hyperparameter_path}")
if output_path.exists():
    raise FileExistsError(f"Output path already exists: {output_path}")

In [ ]:
dataset = LibriSpeech(source).load_dev_clean().sample(SAMPLE_SIZE)
hyperparameters = list(hyperparameter_path.glob("*.yaml"))

In [ ]:
for idx, hyperparameter in enumerate(hyperparameters):
    print(idx, hyperparameter.name)
    output = output_path / f"{hyperparameter.stem}.json"

    DESCRIPTION = f"""
    USE_TOKEN_SAVER_LOADER: {USE_TOKEN_SAVER_LOADER}
    USE_PROMPT: {USE_PROMPT}
    CHUNK_SIZE: {CHUNK_SIZE}

    HYPERPARAMETER: {hyperparameter}
    OUTPUT_PATH: {output}
    """

    evaluate(
        storage,
        output,
        DESCRIPTION,
        dataset,
        models = ["rt_whisper"],
        test_all = TEST_ALL,
        seed = SEED,
        use_save_loader=USE_TOKEN_SAVER_LOADER,
        use_prompt=USE_PROMPT,
        hyperparameter=hyperparameter,
        chunk_size=CHUNK_SIZE,
    )

print("done")